In [ ]:
# ============================================================
# Criar banco SQLite com dados da API SGS (Banco Central)
# ============================================================

import requests
import pandas as pd
import sqlite3
from datetime import datetime, timedelta # Importar datetime e timedelta

# ------------------------------------------------------------
# 1. Baixar série SGS (exemplo: Selic - código 432)
# ------------------------------------------------------------
# Calcular datas para a janela de 10 anos, conforme sugerido pelo erro da API
today = datetime.now()
ten_years_ago = today - timedelta(days=365 * 10) # Aproximadamente 10 anos
data_inicial_str = ten_years_ago.strftime("%d/%m/%Y")
data_final_str = today.strftime("%d/%m/%Y")

# Construir a URL com os parâmetros de data
url = (
    "https://api.bcb.gov.br/dados/serie/bcdata.sgs.432/dados?"
    f"formato=json&dataInicial={data_inicial_str}&dataFinal={data_final_str}"
)

response = requests.get(url)
data = response.json()

# Verificar se a resposta contém dados (uma lista) ou uma mensagem de erro (um dicionário)
if isinstance(data, list): # Se for uma lista, contém os dados esperados
    # Converter para DataFrame
    df_selic = pd.DataFrame(data)
    df_selic['data'] = pd.to_datetime(df_selic['data'], dayfirst=True)
    df_selic['valor'] = pd.to_numeric(df_selic['valor'])

    print("SGS Selic:")
    print(df_selic.head())

    # ------------------------------------------------------------
    # 2. Criar banco SQLite e exportar tabela
    # ------------------------------------------------------------
    # Se você já tem o banco do Lending Club, pode usar o mesmo arquivo
    # Aqui vou criar um novo para exemplo
    conn = sqlite3.connect("dados_sgs.db")

    # Exportar DataFrame para tabela SQL chamada "selic"
    df_selic.to_sql("selic", conn, index=False, if_exists="replace")

    print("Tabela 'selic' criada no banco SQLite.")

    # ------------------------------------------------------------
    # 3. Exemplo de consulta SQL
    # ------------------------------------------------------------
    query = """
    SELECT strftime('%Y-%m', data) as mes, AVG(valor) as media_selic
    FROM selic
    GROUP BY mes
    ORDER BY mes DESC
    LIMIT 10
    """
    result = pd.read_sql(query, conn)
    print("Média mensal da Selic (últimos 10 meses):")
    print(result)

    # Fechar a conexão com o banco de dados quando não for mais necessária
    conn.close()
    print("Conexão com o banco de dados SQLite fechada.")
else: # Se não for uma lista, é uma mensagem de erro da API
    print("Erro ao obter dados da API:")
    print(data)

SGS Selic:
        data  valor
0 2016-05-08  14.25
1 2016-05-09  14.25
2 2016-05-10  14.25
3 2016-05-11  14.25
4 2016-05-12  14.25
Tabela 'selic' criada no banco SQLite.
Média mensal da Selic (últimos 10 meses):
       mes  media_selic
0  2026-05    14.500000
1  2026-04    14.741667
2  2026-03    14.895161
3  2026-02    15.000000
4  2026-01    15.000000
5  2025-12    15.000000
6  2025-11    15.000000
7  2025-10    15.000000
8  2025-09    15.000000
9  2025-08    15.000000
Conexão com o banco de dados SQLite fechada.


In [ ]:
# Listar tabelas existentes no banco
import sqlite3
import pandas as pd # Adicionar import pandas também para pd.read_sql

conn = sqlite3.connect("dados_sgs.db") # Reabrir a conexão com o banco de dados
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
print(tables)
conn.close() # Fechar a conexão após o uso

    name
0  selic


In [ ]:
# Mostrar colunas da tabela 'selic'
import sqlite3
import pandas as pd

conn = sqlite3.connect("dados_sgs.db") # Reabrir a conexão com o banco de dados
columns = pd.read_sql("PRAGMA table_info(selic);", conn)
print(columns)
conn.close() # Fechar a conexão após o uso

   cid   name       type  notnull dflt_value  pk
0    0   data  TIMESTAMP        0       None   0
1    1  valor       REAL        0       None   0


In [ ]:
# Verificar primeiras linhas
import sqlite3
import pandas as pd

conn = sqlite3.connect("dados_sgs.db") # Reabrir a conexão com o banco de dados
check = pd.read_sql("SELECT * FROM selic LIMIT 5;", conn)
print(check)
conn.close() # Fechar a conexão após o uso

                  data  valor
0  2016-05-08 00:00:00  14.25
1  2016-05-09 00:00:00  14.25
2  2016-05-10 00:00:00  14.25
3  2016-05-11 00:00:00  14.25
4  2016-05-12 00:00:00  14.25
